#Nordstream

In [0]:
%sql
CREATE WIDGET TEXT schema_name DEFAULT "vessel";

In [0]:
%sh
pip install geopandas geodatasets folium

In [0]:
import pandas as pd
import geopandas as gpd
import geodatasets
import folium
import matplotlib.pyplot as plt

## Areas of interest

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW danger_areas
as 
/* Nordstream 2022-09-26
   https://simple.wikipedia.org/wiki/2022_Nord_Stream_pipeline_sabotage 
   5 nautic miles
*/ 
SELECT t.description, t.latitude, t.longitude
     , ST_AsText(ST_Point(t.longitude, t.latitude)) area
      /* returns wrong result: 
     , ST_AsText(ST_Buffer(ST_Point(t.longitude, t.latitude), 5 * 1852)) as area */
FROM ( VALUES ('Danger area 1', 54.876667, 15.41)
            , ('Danger area 2', 55.535, 15.698333 )
            , ('Danger area 3', 55.556667, 15.788333 )
            , ('Danger area 4', 55.540833, 15.779 )
      ) t(description, latitude, longitude)
;

In [0]:
%sql
SELECT * FROM danger_areas;

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
SELECT DISTINCT zone, zone_title_prefix_en FROM ${schema_name}.sea_zones ORDER BY 1; /* see also https://havplan.dk/en/page/info */

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW areas
as
SELECT description, area area FROM danger_areas   
UNION ALL 
SELECT zone_title_prefix_en description , geometry area  
FROM ${schema_name}.sea_zones /* IDENTIFIER(concat(:schema_name, '.sea_zones')) not working for temp view */
WHERE zone IN ('Er5');

In [0]:
%sql
SELECT * FROM areas;

In [0]:
areas_df = spark.sql("SELECT * FROM areas").toPandas() # Convert SQL result to Pandas DataFrame
areas_gdf = gpd.GeoDataFrame(areas_df, geometry=gpd.GeoSeries.from_wkt(areas_df['area']))
areas_gdf.plot(figsize=(6, 6))
plt.show()

In [0]:
m = folium.Map(location=[55.1, 15.3], zoom_start=9, tiles="CartoDB positron")
for _, r in areas_gdf.iterrows():
    sim_geo = gpd.GeoSeries(r["geometry"]).simplify(tolerance=0.001) # Simplify the geometry
    geo_j = sim_geo.to_json()
    geo_j = folium.GeoJson(data=geo_j, style_function=lambda x: {"fillColor": "orange"})
    folium.Popup(r["description"]).add_to(geo_j)
    geo_j.add_to(m)
m

## Ship positions

In [0]:
%sql
SELECT * FROM IDENTIFIER(CONCAT(:schema_name, '.messages')) v WHERE v.yyyy_mm_dd = '2022-09-26' LIMIT 10;

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW messages
AS
/* aiming for performance improvement by preselecting relevant subset (columns / rows)
   any effect?
*/
SELECT x.rec_time, x.mmsi, x.imo, x.latitude, x.longitude, x.nav_status, x.callsign, x.name, x.ship_type
FROM (
      SELECT rec_time, mmsi, imo, latitude, longitude, nav_status, callsign, name, ship_type
          , row_number() over (partition by mmsi, date_format(rec_time, 'yyyy-MM-dd HH:mm')  order by rec_time DESC) rnr
      FROM ${schema_name}.messages v
      WHERE v.yyyy_mm_dd = '2022-09-26'
        and v.latitude between 54.0 and 56.0 /* notice simple areas_gdf plot */
        and v.longitude between 14.0 and 16.0
      ) x
WHERE x.rnr = 1 /* only one per minute */
;


In [0]:
%sql
SELECT COUNT(*) FROM messages; /* 124.797 rows */

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW points
as
SELECT v.* /*, ST_AsText(ST_Point(v.longitude, v.latitude)) point_geom*/
FROM messages v 
WHERE exists (SELECT NULL
              FROM areas z
              WHERE ( z.description not like 'Danger%' 
                       and ST_Contains( ST_GeomFromText(z.area), ST_Point(v.longitude, v.latitude)  )
                    )
               or  ( z.description like 'Danger%' 
                     and ST_DistanceSphere( ST_Point(v.longitude, v.latitude), ST_GeomFromText(z.area))
                     < 5*1852 /* danger zone 5 nautic miles, 1 nautic mile = 1852m */
                   )
              )

In [0]:
%sql
SELECT * FROM points ORDER BY mmsi, rec_time;

Databricks visualization. Run in Databricks to view.

In [0]:
points_df = spark.sql("SELECT * FROM points ").toPandas() # Convert SQL result to Pandas DataFrame


In [0]:

for _, r in points_df.iterrows():
    lat = r["latitude"]
    lon = r["longitude"]    
    folium.Marker(
        location=[lat, lon],
        popup=f"""name: {r["name"]} <br> type: {r["ship_type"]} <br> mmsi: {r["mmsi"]}""",
        icon=folium.Icon(color="darkgreen", icon="info-sign"),
    ).add_to(m)

m